# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library. All references to record sets, fields, and columns are made by their `@id` fields, in accordance with Croissant best practices.

### Dataset Source
Source Croissant schema:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print top-level metadata (name, description, etc.)
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"Description: {meta.description}\n")
print(f"Keywords: {meta.keywords}")
print(f"Spatial Coverage: {meta.spatialCoverage}")
print(f"Temporal Coverage: {meta.temporalCoverage}")


## 2. Data Overview

Explore available record sets and fields using their `@id` attributes. This step will help you identify which record sets are available and the structure of each.

> **Note:** To list record sets, we leverage `.recordsets` property, which provides all available record sets and their details.

In [ ]:
# List all record sets and their fields using @id
print('Available record sets and their fields (by @id):\n')
for rs in dataset.recordsets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name','(none)')}")
    print(f"  Description: {rs.get('description','(none)')}")
    if 'field' in rs:
        all_fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields:")
        for fld in all_fields:
            # Fields may be a dict or @id string
            fld_id = fld['@id'] if isinstance(fld, dict) else fld
            print(f"    - {fld_id}")
    else:
        print("  (No fields found)")
    print('-'*45)


### Choose Record Set for Extraction

For demonstration, we'll select the first available record set. Replace `selected_record_set_id` with any desired record set `@id` if needed.

In [ ]:
# Collect all record set @id's for quick access
record_set_ids = [rs['@id'] for rs in dataset.recordsets]
print('Record Set @ids:', record_set_ids)

if not record_set_ids:
    raise Exception('No record sets found in the Croissant metadata.')

# Select a record set for further extraction (here, first one found)
selected_record_set_id = record_set_ids[0]
print(f"Selected Record Set: {selected_record_set_id}")

## 3. Data Extraction
Extract records from the selected record set into a Pandas DataFrame. Fields used in the DataFrame will be identified by their `@id`.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    # Extract records generator for each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Columns (fields @id) in '{selected_record_set_id}':")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Perform typical data processing steps:
- Filter rows by a numeric field (referenced by its `@id`).
- Normalize a numeric field.
- Optionally, group and summarize data.

> **Important:** Replace `numeric_field_id` and `group_field_id` with actual field `@id` values from the DataFrame.

In [ ]:
# Display first few rows to help identify numeric fields (by @id)
display(dataframes[selected_record_set_id].head())

In [ ]:
# Choose a numeric field @id and a group field @id (edit as appropriate based on above display)
numeric_field_id = dataframes[selected_record_set_id].select_dtypes(include=['number']).columns[0] if not dataframes[selected_record_set_id].select_dtypes(include=['number']).empty else dataframes[selected_record_set_id].columns[0]
group_field_id = dataframes[selected_record_set_id].columns[1] if len(dataframes[selected_record_set_id].columns) > 1 else numeric_field_id
print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Group field for EDA: {group_field_id}")

threshold = 10
# Only proceed if the field is numeric
if pd.api.types.is_numeric_dtype(dataframes[selected_record_set_id][numeric_field_id]):
    filtered_df = dataframes[selected_record_set_id][dataframes[selected_record_set_id][numeric_field_id] > threshold]
    print(f"Number of filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First 5 rows with normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group and aggregate if group_field_id exists and is not numeric_field_id
    if group_field_id in filtered_df.columns and group_field_id != numeric_field_id:
        if pd.api.types.is_numeric_dtype(filtered_df[group_field_id]):
            # Don't group if group is numeric and meaningless
            pass
        else:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print(f"Selected field '{numeric_field_id}' is not numeric. Please adjust field selection above.")

## 5. Visualization

Visualize distribution or relationships using field `@id` columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the selected numeric field
if pd.api.types.is_numeric_dtype(dataframes[selected_record_set_id][numeric_field_id]):
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[selected_record_set_id][numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Optional: scatter plot if group_field_id is also numeric or categorical
    if group_field_id in dataframes[selected_record_set_id].columns and group_field_id != numeric_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded the dataset metadata from a Croissant schema URL using `mlcroissant`
- Examined available record sets and fields via their `@id` attributes
- Extracted data for analysis in Pandas
- Performed filtering, normalization, and grouping based on field `@id`s
- Visualized numeric field distributions

> For more advanced analyses, consult the Croissant schema for precise semantics of each `@id`, and refer to [mlcroissant documentation](https://mlcroissant.readthedocs.io/) for extended functionality.